# 07 — PPO with Pixel Observations (Optimized)

PPO agent on Super Mario Bros 1-1 using pixel-based CnnPolicy.

**Why PPO over DQN:**
- No Q-value overestimation / catastrophic forgetting
- On-policy = more stable training
- Better for continuous training (no replay buffer issues)

**Hyperparameters (based on Lior Shilon's article + tuning):**
- Phase 1: 1M steps from scratch (lr=2.5e-4)
- Phase 2: 1M steps from best checkpoint (lr=1e-5)
- Checkpoints every 50k steps + TensorBoard

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    !pip install tensordict torchrl
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_pixel_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create environment
env = make_pixel_env(env_id='SuperMarioBros-1-1-v3')
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

In [ ]:
# Phase 1: Train from scratch
config = PPOConfig(
    policy='CnnPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,         # small entropy bonus for exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=1_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    tensorboard_log='../logs/pixel_ppo',
    verbose=1,
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs/pixel_ppo

In [ ]:
# Phase 1: Train for 1M steps
callback = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo_v2',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,  # log every rollout (every 512 steps)
)
model.save('../models/pixel_ppo_v2/phase1_model')
print('Phase 1 complete!')

In [ ]:
# Phase 2: Load best model, lower lr, train 1M more
from stable_baselines3 import PPO

model = PPO.load('../models/pixel_ppo_v2/phase1_model', env=env)
model.learning_rate = 1e-5  # lower lr for fine-tuning

callback2 = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo_v2',
    save_freq=50_000,
)

model.learn(
    total_timesteps=1_000_000,
    callback=callback2,
    log_interval=1,
)
model.save('../models/pixel_ppo_v2/final_model')
print('Phase 2 complete!')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rewards = callback.episode_rewards + (callback2.episode_rewards if 'callback2' in dir() else [])
lengths = callback.episode_lengths + (callback2.episode_lengths if 'callback2' in dir() else [])
flags = callback.episode_flags + (callback2.episode_flags if 'callback2' in dir() else [])

if len(rewards) > 0:
    window = min(100, len(rewards))

    ax = axes[0]
    ax.plot(rewards, alpha=0.3, color='blue')
    if len(rewards) > window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    ax = axes[1]
    ax.plot(lengths, alpha=0.3, color='orange')
    if len(lengths) > window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    ax = axes[2]
    if len(flags) > 0:
        cumulative_flags = np.cumsum(flags) / (np.arange(len(flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../models/pixel_ppo_v2/training_curves.png', dpi=150)
    plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate
from src.utils.evaluation import evaluate_agent

eval_env = make_pixel_env(env_id='SuperMarioBros-1-1-v3')
results = evaluate_agent(model, eval_env, n_episodes=30)
print(f"Mean reward: {results['mean_reward']:.1f} +/- {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")